# 02 Q学習とSARSA — 迷路の「正解」を、試行錯誤で見つける

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 迷路の攻略メモ（Q表）を、試行錯誤で作っていく回です。
> - **たとえ話**: 歩いた結果をメモして、「ここでは右が得」と書き足していく。何千回も繰り返す。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-3（価値）・2-5（Q学習のしくみ）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

## ゴール
- **Q学習**と**SARSA**で、迷路の最適な行動を、**迷路の仕組みを知らずに**（`step()`だけで）学ぶ。
- 学習の進み方を、**方策の矢印**と**学習曲線**で見る。
- 崖で、2つの方法が**違う方策**を学ぶ理由（オフポリシーとオンポリシー）を理解する。
- **学習中の成績**と**学習後の方策の成績**が、違うことを理解する。

## 前のNotebookとのつながり
00 では、動的計画法で**最適方策（正解）**を求めました。01 では、探索と活用（ε-greedy）を学びました。
ここでは、迷路の仕組み（遷移の確率）を**知らない**エージェントが、ε-greedyで探索しながら、**正解に近づけるか**を調べます。

## 仕組み
各（状態 s、行動 a）について、「そこで行動して、その後うまくやったときのリターンの見込み」**Q(s,a)** を、**表**で持ちます。
1歩進むたびに、次の式で表を更新します。

```
Q(s,a) ← Q(s,a) + α × ( 目標 − Q(s,a) )
```

- **Q学習**: 目標 = r + γ × max Q(s', ·)　（次の状態で、**最良の行動**をすると仮定）
- **SARSA**: 目標 = r + γ × Q(s', a')　（次の状態で、**実際に選ぶ行動 a'**＝探索も含む）

**Q(s,a) が分かれば、各状態で Q が最大の行動を選ぶ**ことが、良い方策です。

## キーワード
- **Q表**: 状態×行動の、価値の表。
- **学習率 α**: 目標に、どれだけ近づけるか（1に近いほど、新しい情報を重視）。
- **オフポリシー（Q学習）**: 実際の行動（探索を含む）とは**別に**、「最良の行動をする方策」の価値を学ぶ。
- **オンポリシー（SARSA）**: **実際に行動している方策**（探索を含む）の価値を学ぶ。
- **学習中の成績**: 探索しながら動いたときの成績。探索のせいで、悪く見える。
- **学習後の方策の成績**: 探索を**やめて**（Qが最大の行動だけ選ぶ）動かした成績。学習とは**別の乱数**で評価する。
- **ゴールに着けない方策**: 探索なしにすると、壁にぶつかり続けるなどで、ゴールに着けない方策になること。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from rl_envs import make_env, value_iteration, evaluate_policy, optimal_q
from rl_envs.viz import draw_grid
from tabular import train, greedy_policy, optimal_action_rate
import run_phase2

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

env = make_env("maze", seed=0)
V_opt, pi_opt = value_iteration(env)
fig, axes = plt.subplots(1, 5, figsize=(17, 3.2))
for ax, n in zip(axes, [1, 10, 30, 100, 500]):
    res = train(make_env("maze", seed=0), "q", episodes=n, alpha=0.5, eps=0.1, seed=0)
    Q = res["Q"]
    r = evaluate_policy(env, greedy_policy(Q), 5, seed=2_000_000)["mean_return"]
    draw_grid(env, ax, V=Q.max(axis=1), policy=greedy_policy(Q), title=f"{n}エピソード後\n方策の成績 {r:.0f}（最適 {V_opt[env.start]:.0f}）")
fig.suptitle("Q学習が進む様子（maze。矢印＝Qが最大の行動、色＝max Q。0は初期値＝まだ試していない）", y=1.12)
fig.tight_layout()

**何をしているか**: Q学習で、迷路（maze）を学習する様子を、1・10・30・100・500エピソード後の**方策（矢印）**で見ます。色は、各マスの max Q（その状態の価値の見込み）です。

**出力の見方**:
- **1エピソード後**: ほとんどのマスが、まだ初期値（Q=0）のまま。方策は、ほぼでたらめで、ゴールには着けません（成績は −200 付近＝200歩で打ち切り）。
- **10〜30エピソード後**: ゴールに近いマスから、価値が伝わって（色が変わって）いきます。報酬はゴールで**初めて**意味を持つので、**ゴール付近から、スタートへ向かって**少しずつ伝わります。
- **100エピソード後**: スタートからゴールまでの道筋に、**正しい矢印**が並びます。成績は、最適（−12）に届きます。
- 道筋から外れたマス（袋小路など）は、あまり訪れないので、矢印が不正確なままです。**エージェントが行く場所だけ、正確に学ぶ**ためです。

In [ ]:
df, curves = run_phase2.run_all(verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(15, 6.4))
def smooth(x, k=10):
    return np.convolve(x, np.ones(k) / k, mode="valid")
for col, label in enumerate(["room", "maze", "cliff"]):
    for mname, c in [("Q学習", "C0"), ("SARSA", "C1")]:
        R = curves[(label, mname)]["returns"]
        E = curves[(label, mname)]["eval"]
        sm = np.array([smooth(r) for r in R])
        axes[0][col].plot(np.arange(sm.shape[1]) + 10, sm.mean(0), color=c, label=mname)
        axes[0][col].fill_between(np.arange(sm.shape[1]) + 10, sm.mean(0) - sm.std(0), sm.mean(0) + sm.std(0), color=c, alpha=0.15)
        xs = np.arange(1, E.shape[1] + 1) * 10
        axes[1][col].plot(xs, np.median(E, axis=0), color=c, label=mname + "（中央値）")
        axes[1][col].fill_between(xs, np.percentile(E, 10, axis=0), np.percentile(E, 90, axis=0), color=c, alpha=0.15)
    v = df[(df["環境"] == label)]["最適な価値"].iloc[0]
    for r in (0, 1):
        axes[r][col].axhline(v, color="k", ls=":", lw=1, label="最適")
    lo = {"room": -60, "maze": -80, "cliff": -150}[label]
    axes[0][col].set(title=f"{label}: 学習中の成績（探索あり）", ylim=(lo, 0), xlabel="エピソード")
    axes[1][col].set(title=f"{label}: 学習後の方策の成績（探索なし）", ylim=(lo, 0), xlabel="エピソード")
    for r in (0, 1):
        axes[r][col].grid(alpha=0.3); axes[r][col].legend(fontsize=7)
fig.suptitle("10シードの平均（上は平均±標準偏差、下は中央値と10〜90%範囲）")
fig.tight_layout()

**何をしているか**: 3つの迷路で、Q学習（青）とSARSA（橙）を、10個のシードで学習させます。
上段は**学習中の成績**（探索ありで動いたときのリターン）、下段は**学習後の方策の成績**（10エピソードごとに、探索なしで評価。学習とは別の乱数）です。点線は最適。

**出力の見方**:
- **room・maze**: 学習中の成績は、**数十エピソードで最適近く**まで上がります。ただし、最適より少し悪いまま（room −8.84 対 最適 −8）。**探索（ε=0.1）で、ときどきランダムに動く**ためです。
- 下段（探索なし）: **Q学習は、数十エピソードで最適（点線）に達して、安定**します。**SARSAは、中央値は最適**ですが、**橙の影が下に何度も伸びます**。シードによって、**ときどき、ゴールに着けない方策**（成績 −200 付近）になるためです（後で調べます）。
- **cliff（右列）**: 学習中の成績は、**Q学習の方が、SARSAよりずっと悪く**、大きく振れます（後で調べます）。一方、下段（探索なし）では、**Q学習が最適（−13）**に達し、SARSAは**最適より遠回り（−17付近）**で止まります。

In [ ]:
res = {}
for mname in ["q", "sarsa"]:
    res[mname] = train(make_env("cliff", seed=0), mname, episodes=500, alpha=0.5, eps=0.1, seed=0)
envc = make_env("cliff")
Vc, pic = value_iteration(envc)
fig, axes = plt.subplots(3, 1, figsize=(9.5, 7.2))
draw_grid(envc, axes[0], V=Vc, policy=pic, title=f"最適方策（動的計画法）: スタートの価値 {Vc[envc.start]:.0f}")
for ax, (mname, label) in zip(axes[1:], [("q", "Q学習"), ("sarsa", "SARSA")]):
    Q = res[mname]["Q"]
    r = evaluate_policy(envc, greedy_policy(Q), 50, seed=2_000_000)["mean_return"]
    draw_grid(envc, ax, V=Q.max(axis=1), policy=greedy_policy(Q), title=f"{label}が学習した方策（500エピソード後）: 探索なしの成績 {r:.0f}")
fig.tight_layout()

**何をしているか**: 崖（cliff）で、Q学習とSARSAが学習した**方策（矢印）**を、最適方策と並べます。

**出力の見方**:
- **最適方策（上）**: 崖の**すぐ上の段**を進む、最短ルート（−13）。
- **Q学習**: ほぼ同じ、**崖沿いの最短ルート**を学びます。
- **SARSA**: 崖から**離れた段（上の方）**を進む、**遠回りで安全なルート**を学びます。
- なぜ違うのか: 学習中は、ε=0.1で**ときどきランダムに動く**ので、崖沿いを歩くと、**うっかり崖に落ちる**ことがあります。
  - **Q学習**は、「次は最良の行動をする」と仮定して価値を学ぶので、**探索の事故を価値に含めません**。だから、崖沿いが最短で、最善と学びます（**オフポリシー**）。
  - **SARSA**は、**実際に探索しながら行動する方策**の価値を学ぶので、「崖沿いは、ときどき落ちて、−100になる」ことを**価値に反映**します。だから、崖から離れた安全なルートを選びます（**オンポリシー**）。

In [ ]:
tbl = df[df["環境"] == "cliff"][["手法", "学習中の成績(最後100回)", "学習後の方策の成績", "ゴールに着けない方策(シード数)", "崖に落ちた回数/エピソード(最後100)"]]
tbl.set_index("手法")

**何をしているか**: 崖での、4つの方法（Q学習・SARSA、それぞれ探索を減らす/減らさない）の結果を、10シードでまとめます。

**出力の見方**（探索率は、固定の0.1の行）:
- **Q学習**: 学習後の方策の成績は**最適（−13）**。しかし、**学習中の成績は −49**と悪く、**1エピソードに平均0.33回、崖に落ちて**います。**最短ルートを知っていても、探索中に崖に落ちる**のです。
- **SARSA**: 学習中の成績は **−27**で、Q学習より**ずっと良く**、崖に落ちる回数も**0.05回**と少ない。ただし、学習後の方策は、**最適より遠回り**（−17前後。下のシード1つは失敗）。
- つまり、**どちらが良いかは、目的による**: **学習中も安全に動きたい**（現実のロボットなど、失敗が痛い）ならSARSA、**学習後の最適な方策**が欲しい（シミュレータで学習する）ならQ学習。

In [ ]:
# SARSAの失敗例: 学習後の方策（探索なし）がゴールに着けない（room, シード1000009）
envr = make_env("room", seed=1_000_009)
r = train(envr, "sarsa", episodes=500, alpha=0.5, eps=0.1, seed=1_000_009)
pol = greedy_policy(r["Q"])
ev = evaluate_policy(envr, pol, 20, seed=2_000_000)
r2 = train(make_env("room", seed=1_000_009), "sarsa", episodes=500, alpha=0.5, eps=0.3, eps_decay=0.99, eps_min=0.01, seed=1_000_009)
pol2 = greedy_policy(r2["Q"])
ev2 = evaluate_policy(envr, pol2, 20, seed=2_000_000)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
draw_grid(envr, axes[0], V=r["Q"].max(axis=1), policy=pol, title=f"探索率 0.1 で固定: 成績 {ev['mean_return']:.0f}（ゴール到達 {ev['success_rate']:.0%}）")
draw_grid(envr, axes[1], V=r2["Q"].max(axis=1), policy=pol2, title=f"探索率を 0.3→0.01 に減らす: 成績 {ev2['mean_return']:.0f}（ゴール到達 {ev2['success_rate']:.0%}）")
fig.tight_layout()

**何をしているか**: SARSAが、**学習後の方策（探索なし）で失敗する例**を見ます。左は探索率を0.1で固定、右は学習が進むほど探索を減らした（0.3→0.01）場合です。

**出力の見方**:
- **左**: 方策の矢印をたどると、**壁に向かう矢印（その場にとどまる行動）**に着き、**ずっと抜け出せません**（成績 −200、ゴール到達0%）。
- なぜか: SARSAは、**探索しながら行動する方策**の価値を学びます。「壁に向かって、その場にとどまる」行動も、探索（10%でランダム）のおかげで、**平均10歩ほどで抜け出せる**ので、価値が**それほど悪くならず**、他の行動と**競り合って**しまいます。ところが、探索をやめた方策では、**永遠に抜け出せません**。
- **右**: 学習が進むにつれて探索を減らす（0.3→0.01）と、このような方策は**起きません**。学習中の成績も、最適近く（−8.1）に上がります。
- Q学習は、「次は最良の行動」と仮定するので、壁にとどまる行動の価値が**どんどん下がって**避けられ、この失敗は起きません（10シードで0回）。
- 教訓: **オンポリシーの価値は、探索を続ける前提**。探索なしの方策で使うなら、**探索を減らす**（または、最後は探索をやめて評価する）工夫が要る。

In [ ]:
cols = ["環境", "手法", "最適な価値", "学習中の成績(最後100回)", "学習後の方策の成績", "ゴールに着けない方策(シード数)", "最適な行動の割合"]
df[cols].set_index(["環境", "手法"])

**何をしているか**: 4つの環境 × 4つの方法を、10シードでまとめた表です（500エピソード、学習率α=0.5）。「学習後の方策の成績」は、探索なしで100回動かした平均（学習とは別の乱数）です。

**出力の見方**:
- **room・maze（すべらない）**: Q学習は、**学習後の方策が最適（−8、−12）**に毎回達します（ばらつき0）。SARSA（探索固定）は、**1シードが失敗**して、平均が悪化（−27、−31）。探索を減らすと、全シードで最適に達します。
- **最適な行動の割合**（maze 0.76）: 全マスのうち、最適な行動を選べているマスの割合。**道筋から外れたマスは、訪れないので、学習されません**（0.76で止まるのは自然）。大事なのは、**スタートからゴールへの道筋**が最適なこと。
- **cliff（すべる0.2）**: Q学習も、最適（−29.8）には届きません（探索を減らして −33）。**確率的な環境では、学習率が大きい（α=0.5）と、更新がぶれて、500エピソードでは足りません**。学習率を下げて、エピソードを増やす必要があります（演習）。

## 試してみよう（演習）
1. **学習率を変える**: `src/run_phase2.py` の `ALPHA = 0.5` を `0.1` と `1.0` にして、`python src/run_phase2.py` を実行してください。見るポイントは、cliff（すべる0.2）の成績と、SARSAの失敗シード数です。分かるのは、すべる環境では、学習率が小さいほど安定する（その代わり、学習が遅い）ことです。
2. **エピソード数を増やす**: `EPISODES = 500` を `3000` にして、cliff（すべる0.2）で、Q学習が最適（−29.77）に近づくか確かめてください。
3. **探索率を変える**: `METHODS` の `eps=EPS`（0.1）を `0.3` と `0.02` に変えてください。見るポイントは、cliff での、Q学習の「崖に落ちた回数」と、学習後の方策の成績です。分かるのは、探索が多いと学習中の失敗が増える一方、少ないと最適な方策を見つけにくいことです。
4. **初期値を変える**: `train(..., q0=0.0)` の `q0` を `-50`（悲観的）と `10`（楽観的）にして、学習の速さを比べてください。報酬は負なので、`q0=0` は**すでに楽観的**です（試したことのない行動が、有利に見える）。

## まとめ
- **Q学習・SARSA**は、迷路の仕組みを知らず、`step()`だけで、**最適な方策に近づける**。報酬は**ゴール付近から**、スタートへ伝わる。
- **学習中の成績**（探索あり）と、**学習後の方策の成績**（探索なし）は**別**。崖では、Q学習は、学習中は−49と悪いのに、学習後は最適（−13）。
- **オフポリシー（Q学習）**は最適な方策を、**オンポリシー（SARSA）**は探索を含めた安全な方策を学ぶ。目的で選ぶ。
- **オンポリシーの方策は、探索なしだと失敗**することがある（壁にとどまるループ）。**探索を減らす**と防げる。
- すべる環境では、**学習率を下げ、エピソードを増やす**必要がある。

次の Phase3 では、**報酬の設計**・**割引率**・**探索率**を変えて、強化学習が**思わぬ振る舞い**をする落とし穴を調べます。